# Practice: classes, dataclasses and enums

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python-05-practice-classes-and-dataclasses.ipynb)

Companion to the [Classes, dataclasses and enums](python-05-classes-and-dataclasses.ipynb) chapter. The exercises cover instance state, properties, frozen dataclasses, enums, abstract base classes and equality with hashing.

## How to use this notebook

Work through the exercises in order. Each one has three parts:

1. **Question:** what to build, with the rules it must follow.
2. **Exercise cell:** replace the `pass` with your code. Keep the function name and parameters.
3. **Check cell:** run it after your code. Each line says `pass` when your function gives the expected result, and `todo` when it does not yet. Checks never stop the notebook, so you can run everything at any time.

The last cell prints your score. Use `Runtime -> Run all` to rerun everything after a change.

In [ ]:
_tally = {"passed": 0, "total": 0}

def check(label, make_actual, expected):
    """Run one check. It never raises, so the notebook always runs from top to bottom."""
    _tally["total"] += 1
    try:
        actual = make_actual()
    except Exception as error:
        print(f"error  {label}: {type(error).__name__}: {error}")
        return
    if actual == expected:
        _tally["passed"] += 1
        print(f"pass   {label}")
    else:
        print(f"todo   {label}: got {actual!r}")

def raises(make_actual, error_type):
    """True when the call raises error_type, False when it raises anything else or nothing."""
    try:
        make_actual()
    except error_type:
        return True
    except Exception:
        return False
    return False

def score():
    print(f"\nScore: {_tally['passed']} of {_tally['total']} checks pass")

## Exercise 1: a stack class

Write a class `Stack` with these methods:

- `push(item)` adds an item to the top.
- `pop()` removes and returns the top item. An empty stack raises `IndexError`.
- `peek()` returns the top item without removing it.
- `__len__` returns the number of items, so `len(stack)` works.

*Hint: keep the items in a list in `self`, and let the list's own `pop` and `[-1]` do the work.*

In [ ]:
class Stack:
    # your code here
    pass

In [ ]:
check(
    "Stack: pop returns the last pushed item",
    lambda: (lambda s: (s.push(1), s.push(2), s.pop())[-1])(Stack()),
    2,
)
check(
    "Stack: peek and len leave the item in place",
    lambda: (lambda s: (s.push("a"), s.peek(), len(s)))(Stack())[1:],
    ("a", 1),
)
check(
    "Stack: pop on empty raises IndexError",
    lambda: raises(lambda: Stack().pop(), IndexError),
    True,
)

## Exercise 2: a temperature with a property and validation

Write a class `Temperature`. Its constructor takes `celsius`. Raise `ValueError` when the value is below absolute zero, which is `-273.15`. Add a read-only property `fahrenheit` that returns `celsius * 9 / 5 + 32`.

In [ ]:
class Temperature:
    # your code here
    pass

In [ ]:
check("Temperature(100).fahrenheit", lambda: Temperature(100).fahrenheit, 212.0)
check("Temperature(0).fahrenheit", lambda: Temperature(0).fahrenheit, 32.0)
check(
    "Temperature(-300) raises ValueError",
    lambda: raises(lambda: Temperature(-300), ValueError),
    True,
)

## Exercise 3: a frozen product with a discount method

Write a frozen dataclass `Product` with two fields, `name` (a string) and `price_cents` (an integer). Add a method `with_discount(percent)` that returns a **new** `Product` whose price is reduced by `percent`, using whole cents with the remainder dropped. The original product must not change.

*Hint: `dataclasses.replace` builds the changed copy, or you can call `Product(...)` directly.*

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Product:
    name: str
    price_cents: int

    def with_discount(self, percent):
        # your code here
        pass

In [ ]:
check(
    "Product.with_discount(10) price",
    lambda: Product("tea", 1000).with_discount(10).price_cents,
    900,
)
check(
    "Product.with_discount leaves the original unchanged",
    lambda: (lambda p: (p.with_discount(10), p.price_cents)[1])(Product("tea", 1000)),
    1000,
)
check(
    "Product is frozen",
    lambda: raises(lambda: setattr(Product("x", 1), "price_cents", 2), AttributeError),
    True,
)

## Exercise 4: an enum and a function that uses it

Define an `Enum` called `Priority` with the members `LOW = 1`, `MEDIUM = 2` and `HIGH = 3`. Then write `highest_name(priorities)`. It takes a list of `Priority` members and returns the **name** of the member with the highest value, as a string.

In [ ]:
from enum import Enum

class Priority(Enum):
    # your members here
    pass

def highest_name(priorities):
    # your code here
    pass

In [ ]:
check(
    "highest_name([LOW, HIGH, MEDIUM]) == 'HIGH'",
    lambda: highest_name([Priority.LOW, Priority.HIGH, Priority.MEDIUM]),
    "HIGH",
)
check("Priority.MEDIUM.value == 2", lambda: Priority.MEDIUM.value, 2)

## Exercise 5: a subclass that implements an abstract method

`Exporter` is given below. It is an abstract base class with one abstract method, `export(rows)`. Write `CsvExporter`, a subclass that implements `export`. It joins each row's values with commas, and joins the rows with newlines.

In [ ]:
from abc import ABC, abstractmethod

class Exporter(ABC):
    @abstractmethod
    def export(self, rows):
        """Return the rows as one string."""

class CsvExporter(Exporter):
    # your code here
    pass

In [ ]:
check(
    "CsvExporter().export(...)",
    lambda: CsvExporter().export([["a", "b"], ["c", "d"]]),
    "a,b\nc,d",
)
check(
    "Exporter cannot be created directly",
    lambda: raises(lambda: Exporter(), TypeError),
    True,
)

## Exercise 6: equality and hashing for a point

Write a class `Point` with `x` and `y` attributes. Two points are equal when both coordinates match. A point must also be hashable, so it can be a dictionary key. Define both `__eq__` and `__hash__`.

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    # your __eq__ and __hash__ here

In [ ]:
check("Point(1, 2) == Point(1, 2)", lambda: Point(1, 2) == Point(1, 2), True)
check("Point(1, 2) != Point(2, 1)", lambda: Point(1, 2) != Point(2, 1), True)
check(
    "Point works as a dictionary key",
    lambda: {Point(1, 2): "a"}[Point(1, 2)],
    "a",
)

## Score

In [ ]:
score()